In [13]:
!pip install -q openai tqdm
import json, re, time, random
from collections import defaultdict, Counter
from google.colab import userdata
from openai import OpenAI
from tqdm import tqdm

client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
MODEL = "gpt-4o"

N     = 4       # 시드 1건당 변형 수. train 35 → 35 + 35×4 = 175 → QC 후 ~150
TEMP  = 0.9

In [15]:
# 시드 로드 + train/valid 분할
raw = json.load(open('pep_seed_golden.json', encoding='utf-8'))
seed_all = raw['data'] if isinstance(raw, dict) and 'data' in raw else raw

by_label = defaultdict(list)
for r in seed_all:
    by_label[r['output']['label']].append(r)

random.seed(42)
seed_train, seed_valid = [], []
VALID_RATIO = 0.30
for label, items in by_label.items():
    random.shuffle(items)
    n_valid = max(1, round(len(items) * VALID_RATIO))
    seed_valid += items[:n_valid]
    seed_train += items[n_valid:]

random.shuffle(seed_train); random.shuffle(seed_valid)
print(f"train {len(seed_train)} / valid {len(seed_valid)}")
print("train:", dict(Counter(r['output']['label'] for r in seed_train)))
print("valid:", dict(Counter(r['output']['label'] for r in seed_valid)))

json.dump(seed_valid, open('PEP_valid.json','w'), ensure_ascii=False, indent=2)
print("저장: PEP_valid.json (순수, 증강 안 함)")

시드 개수: 30


In [ ]:
# 증강 프롬프트
PEP_AUG_PROMPT = """..."""   # ← 네가 보낸 프롬프트 전문

In [18]:
# 증강 호출 + 파서
def parse_json(t):
    t = re.sub(r"^```(json)?|```$", "", t.strip(), flags=re.M).strip()
    return json.loads(t)

def augment(rec):
    prompt = PEP_AUG_PROMPT.replace("{N}", str(N)) \
             .replace("{여기에 골든 1건}", json.dumps(rec, ensure_ascii=False))
    resp = client.chat.completions.create(
        model=MODEL, temperature=TEMP,
        messages=[{"role":"user","content":prompt}],
    )
    return parse_json(resp.choices[0].message.content)

In [19]:
# QC
VALID = {"충족","불가","검토"}
def qc(rec, orig):
    if not {"id","input","output"}.issubset(rec.keys()): return False, "필드누락"
    inp, out = rec["input"], rec["output"]
    if not {"criteria","rfp_excerpt","pep_excerpt"}.issubset(inp.keys()): return False, "input누락"
    if out.get("label") not in VALID: return False, "label이상"
    # 원본과 label·criteria 동일 유지 확인
    if out["label"] != orig["output"]["label"]: return False, "label변경됨"
    if inp["criteria"] != orig["input"]["criteria"]: return False, "criteria변경됨"
    ev = out.get("eval")
    if not isinstance(ev, list) or not ev: return False, "eval형식"
    crit = set(inp["criteria"])
    for line in ev:
        feat = line.split(":")[0].strip()
        if feat not in crit: return False, f"criteria밖특성:{feat}"
        # 정확성 검토 금지
        head = line.split("—")[0]
        if feat == "정확성" and "검토" in head: return False, "정확성검토금지"
    return True, ""

In [20]:
# 실행(train 시드만 변형)
random.seed(42)
aug, dropped = [], []
for rec in tqdm(seed_train):          # ← train만!
    try:
        variants = augment(rec)
    except Exception as e:
        dropped.append({"id": rec["id"], "err": str(e)}); continue
    for i, v in enumerate(variants):
        v["id"] = f"{rec['id']}-aug{i+1}"       # id 부여
        ok, why = qc(v, rec)
        if not ok:
            dropped.append({"id": v.get("id"), "reason": why}); continue
        aug.append(v)

train_full = seed_train + aug
json.dump(train_full, open('PEP_train.json','w'), ensure_ascii=False, indent=2)
print(f"train시드 {len(seed_train)} + 증강 {len(aug)} = {len(train_full)} / 탈락 {len(dropped)}")
print("label 분포:", dict(Counter(r['output']['label'] for r in train_full)))
if dropped: print("탈락 샘플:", dropped[:5])